# Prepare dataset and KG

## Load knowledge base data 

Read tiples in .txt and load them as a dictionary

In [1]:
#Download dataset from: https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE?resourcekey=0-kdv6ho5KcpEXdI2aUdLn_g
dataset_folder = ""
dataset_file = dataset_folder + "METAQA/kb.txt"

In [2]:
with open(dataset_file) as file:
    lines = [line.rstrip().split('|') for line in file]

In [3]:
movies = {}

for l in lines:
    movie_id, relation, person = l  # unpack for readability

    # --- Ensure movie exists ---
    if movie_id not in movies:
        movies[movie_id] = {}
    movie_dict = movies[movie_id]

    # --- Add to movie dictionary ---
    movie_dict.setdefault(relation, []).append(person)


## Create Neo4j graph


### Run Neo4J server

In [4]:
from neo4j import GraphDatabase

# Server details
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "neo4j123")
DATABASE_NAME = "neo4j"

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection with the server stablished successfully.")

Connection with the server stablished successfully.


### Wipe out current graph

### Populate Movies graph

In [5]:
#Attributes as lists
def add_movies(tx, movies):
    for name, attrs in movies.items():
        query = """
        MERGE (m:Movie {name: $name})
        SET m += $props
        """
        tx.run(query, name=name, props=attrs)

def add_relationships(tx):
    # Directors
    tx.run("""
    MATCH (m:Movie)
    UNWIND m.directed_by AS director_name
    MERGE (p:Director {name: director_name})
    MERGE (m)-[:DIRECTED_BY]->(p)
    """)
    
    # Actors
    tx.run("""
    MATCH (m:Movie)
    UNWIND m.starred_actors AS actor_name
    MERGE (p:Actor {name: actor_name})
    MERGE (p)-[:STARRED_IN]->(m)
    """)
    
    # Writers
    tx.run("""
    MATCH (m:Movie)
    UNWIND m.written_by AS writer_name
    MERGE (p:Writer {name: writer_name})
    MERGE (m)-[:WRITTEN_BY]->(p)
    """)
    
    # Genres
    tx.run("""
    MATCH (m:Movie)
    UNWIND m.has_genre AS genre_name
    MERGE (g:Genre {name: genre_name})
    MERGE (m)-[:HAS_GENRE]->(g)
    """)
    
    # Languages
    tx.run("""
    MATCH (m:Movie)
    UNWIND m.in_language AS language_name
    MERGE (l:Language {name: language_name})
    MERGE (m)-[:IN_LANGUAGE]->(l)
    """)

### Query example

In [6]:
driver = GraphDatabase.driver(URI, auth=AUTH)

records, summary, keys = driver.execute_query("""
    MATCH (n:Genre)
    RETURN n.name
    LIMIT 2
    """,
    database_=DATABASE_NAME,
)

# Loop through results and do something with them
for record in records:
    print(record.data())  # obtain record as dict

driver.close()

{'n.name': 'War'}
{'n.name': 'Drama'}


## Prepare Q&A data

In [7]:
def load_qa_data(file):
    questions = []
    answers = []
    
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()  # remove newline/whitespace
            if not line:  # skip empty lines
                continue
            
            # split into question and answer part
            q, a = line.split("\t", 1)
            
            # store question
            questions.append(q)
            
            # split answers by "|" and strip whitespace
            answers.append([ans.strip() for ans in a.split("|")])
    return questions, answers 

In [8]:
#Download dataset from: https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE?resourcekey=0-kdv6ho5KcpEXdI2aUdLn_g
test_file1 = dataset_folder+"METAQA/1-hop/qa_test.txt"
test_file2 = dataset_folder+"METAQA/2-hop/qa_test.txt"
test_file3 = dataset_folder+"METAQA/3-hop/qa_test.txt"

test_questions1, test_answers1 = load_qa_data(test_file1)
test_questions2, test_answers2 = load_qa_data(test_file2)
test_questions3, test_answers3 = load_qa_data(test_file3)

In [9]:
print('Number of test questions:', len(test_questions1) + len(test_questions2) + len(test_questions3))print('Number of test questions:', len(test_questions1) + len(test_questions2) + len(test_questions3))

Number of test questions: 39093


### Get unique questions and parse them

In [10]:
import re

def escape_quotes_in_brackets(text):
    def replacer(match):
        inside = match.group(1)
        # Escape all single quotes inside the brackets
        return "[" + inside.replace("'", r"\'") + "]"
    
    # Find [ ... ] groups and apply replacer
    return re.sub(r"\[([^\]]*)\]", replacer, text)

def questions_stats(test_question, test_answer):
    questions = {}
    unique_questions = []
    unique_answers = []
    for question, answer in zip(test_question, test_answer):
        unique_question =  re.sub(r"\[.*?\]", "", question)
    
        if unique_question in questions:
            questions[unique_question] += 1
        else:
            questions[unique_question] = 1
            # Add spaces around only , . ! ?
            text_spaced = re.sub(r',(?![^\[]*\])', r' , ', question)
            # Remove extra spaces
            text_spaced = re.sub(r'\s+', ' ', text_spaced).strip()
            unique_questions.append(text_spaced)
            unique_answers.append(answer)
    
    less_frequent = len(test_question)
    more_frequent = 0
    for q in questions:
        if questions[q] < less_frequent:
            less_frequent = questions[q]
        if questions[q] > more_frequent:
            more_frequent = questions[q]

    return ({
        'len_questions' : len(test_question),
        'unique_questions' : len(questions),
        'less_frequent' : less_frequent,
        'more_frequent' : more_frequent
    }, unique_questions, unique_answers)    

In [11]:
stats1, unique_questions1, unique_answers1 = questions_stats(test_questions1, test_answers1)
stats2, unique_questions2, unique_answers2 = questions_stats(test_questions2, test_answers2)
stats3, unique_questions3, unique_answers3 = questions_stats(test_questions3, test_answers3)

sample_questions = unique_questions1 + unique_questions2 + unique_questions3
sample_answers = unique_answers1 + unique_answers2 + unique_answers3

print("Number of unique questions:", len(sample_questions))

Number of unique questions: 503


## Get Graph Structure

In [12]:
# Your Cypher query
cypher_query = """
CALL apoc.meta.schema() YIELD value
WITH apoc.convert.toMap(value) AS schema
UNWIND keys(schema) AS label
WITH label, schema[label].properties AS props
RETURN label, apoc.convert.toJson(props) AS properties_json
"""

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session() as session:
    result = session.run(cypher_query)
    graph_structure = [record.data() for record in result]
driver.close()

# Prepare BAF agent

In [13]:
# Apply patch to besser/agent/nlp/ner/simple_ner.py 
# This patch allo us to flag sensitive entities using brackets

from pathlib import Path
project_root = str(Path().resolve())  # current working dir
patch_file = project_root + '/simple_ner.patch'

import besser.agent.nlp.ner.simple_ner
file_to_patch = besser.agent.nlp.ner.simple_ner.__file__

!patch --batch {file_to_patch} < {patch_file}

2026-01-14 10:55:58,457 - WARNING - langchain dependencies in RAG could not be imported. You can install them from the requirements/requirements-extras.txt file


patching file /home/su_dalle-lucca-tosi/.pyenv/versions/3.11.0/envs/baf/lib/python3.11/site-packages/besser/agent/nlp/ner/simple_ner.py
Reversed (or previously applied) patch detected!  Assuming -R.
Hunk #4 FAILED at 138.
1 out of 5 hunks FAILED -- saving rejects to file /home/su_dalle-lucca-tosi/.pyenv/versions/3.11.0/envs/baf/lib/python3.11/site-packages/besser/agent/nlp/ner/simple_ner.py.rej


In [14]:
import json

from besser.agent.nlp.preprocessing.text_preprocessing import process_text, tokenize
from besser.agent.core.agent import Agent
from besser.agent.core.session import Session
from besser.agent.exceptions.logger import logger
from besser.agent.nlp.intent_classifier.intent_classifier_prediction import IntentClassifierPrediction
from besser.agent.library.entity.base_entities import number_entity, datetime_entity, any_entity
from besser.agent.nlp.intent_classifier.intent_classifier_configuration import IntentClassifierConfiguration
from besser.agent.nlp.intent_classifier.simple_intent_classifier_pytorch import SimpleIntentClassifierTorch

2026-01-14 10:56:01,073 - WARNING - keras dependencies in SimpleIntentClassifierTF could not be imported. You can install them from the requirements/requirements-tensorflow.txt file
2026-01-14 10:56:01,075 - WARNING - librosa dependencies in HFSpeech2Text could not be imported. You can install them from the requirements/requirements-extras.txt file
2026-01-14 10:56:01,076 - WARNING - transformers dependencies in HFSpeech2Text could not be imported. You can install them from the requirements/requirements-llms.txt file
2026-01-14 10:56:01,077 - WARNING - speech_recognition dependencies in APISpeech2Text could not be imported. You can install them from the requirements/requirements-extras.txt file
2026-01-14 10:56:01,529 - WARNING - cv2 dependencies in websocket_callbacks.py could not be imported. You can install them from the requirements/requirements-extras.txt file
2026-01-14 10:56:01,530 - WARNING - plotly dependencies in websocket_callbacks.py could not be imported. You can install t

### Define Agent

In [15]:
agent = Agent('metaqa__agent')

# Load agent properties stored in a dedicated file
agent.load_properties('config.ini')

#edit default configurations
ic_config = agent._default_ic_config
ic_config.num_epochs = 10
ic_config.lr = 5e-5
ic_config.input_max_num_tokens=25

# Disable ui
websocket_platform = agent.use_websocket_platform(use_ui=False)

### Define agent states
We only need the initial and the LLM state

In [16]:
# STATES
s0 = agent.new_state('s0', initial=True, ic_config=ic_config)
llm_state = agent.new_state('llm_state')

### Define agent entries
Those are read directly from the KG. 

In this case, we manually define them as the graph is small.

In [17]:
#Node labels
node_label_entries = {}
node_labels = ['Movie', 'Director', 'Actor', 'Writer', 'Genre', 'Language']
for label in node_labels:
    node_label_entries.setdefault(label, [])


#Node properties
node_property_entries = {}
node_properties = ['in_language', 'release_year', 'name', 'has_tags', 'written_by',  
                   'starred_actors', 'has_imdb_votes', 'has_genre', 'has_imdb_rating']
for node_property in node_properties:
    node_property_entries.setdefault(node_property, [])

#Relation labels
relations_entries = {}
relations = ['DIRECTED_BY', 'STARRED_IN', 'WRITTEN_BY', 'HAS_GENRE', 'IN_LANGUAGE']
for relation in relations:
    relations_entries[relation] = []

#Node sensitive values
node_values_entries = {}
for movie_name in movies:
    if len(movie_name.split(" ")) > 1: # do not accept movie names with less than 2 words
        node_values_entries[movie_name] = []
    for movie_property in movies[movie_name]:
        for value in movies[movie_name][movie_property]:
            if movie_property == 'has_tags' and len(value.split(" ")) <= 1: # do not accept movie tags with less than 2 words
                continue
            else:    
                node_values_entries[value] = []

#### Enrich entries

In [18]:
node_property_entries['directed_by'] = ['directed by']
node_property_entries['in_language'] = ["language", "original language", "spoken language", "language of the movie", "language used"]
node_property_entries['release_year'] = ["year of release", "release year", "released in", "publication year", "year"]
node_property_entries['has_tags'] = ["tags", "keywords", "tag list", "labels", "topics"]
node_property_entries['written_by'] = ["writer", "screenwriter", "authored by", "script by", "written by"]
node_property_entries['starred_actors'] = ["cast", "starring", "actors", "featured cast", "main cast", "lead actors"]
node_property_entries['has_imdb_votes'] = ["IMDb votes", "vote count", "ratings count", "number of IMDb votes", "IMDb vote total"]
node_property_entries['has_genre'] = ["genre", "genres", "category", "film genre", "type"]
node_property_entries['has_imdb_rating'] = ["IMDb rating", "average rating", "IMDb score", "user rating on IMDb", "rating score"]
node_property_entries['name'] = ["title", "name", "display name", "official title", "label"]

In [19]:
node_label_entries['Movie'] = ["film", "motion picture", "feature film", "movie", "picture"]
node_label_entries['Director'] = ["film director", "director", "helmer", "filmmaker"]
node_label_entries['Actor'] = ["performer", "cast member", "actor/actress", "thespian", "star"]
node_label_entries['Writer'] = ["screenwriter", "scriptwriter", "writer", "author"]
node_label_entries['Genre'] = ["category", "type", "style", "genre", "classification"]
node_label_entries['Language'] = ["language", "tongue", "lingo", "spoken language", "original language"]

In [20]:
relations_entries['DIRECTED_BY'] = ["directed by", "helmed by", "under the direction of", "by director"]
relations_entries['STARRED_IN'] = ["starred in", "acted in", "appeared in", "featured in", "was in the cast of"]
relations_entries['WRITTEN_BY'] = ["written by", "authored by", "screenplay by", "script by"]
relations_entries['HAS_GENRE'] = ["has genre", "belongs to genre", "classified as", "falls under", "of type"]
relations_entries['IN_LANGUAGE'] = ["in language", "in", "spoken in", "originally in", "language is"]

In [21]:
#If a entry appears both in the structure of the text and as a value of the graph, we consider it as part of the graph structure.
# eg. if a movie is called 'Genre', we exclude it from 'node_values_entries' and keep it only in 'node_labels'

from util import resolve_entity_conflicts, replace_placeholders, replace_placeholders_back, extract_bracketed_entities
# After building your original dictionaries, filter them:
filtered_node_values = resolve_entity_conflicts(
    node_values_entries,
    node_property_entries, 
    node_label_entries,
    language='en',
    remove_common_words=True
)

# Replace your original dictionary
node_values_entries = filtered_node_values

Building stemmed properties and labels...
Found 54 unique stemmed properties
Found 29 unique stemmed labels
Filtering node values...

Filtering Summary:
Original values: 38458
Filtered values: 38456
Removed conflicts: 0
Removed common/short: 2

Removed common/short words (showing first 10):
  'Nas' - too short (stemmed: 'na')
  'Em' - too short (stemmed: 'em')


### Define agent entities

In [22]:
# ENTITIES
'''
Node_labels (Classes)-> Themes, Project, Contact, Partner, Dataset, Variable
Node_properties-> name, last_name, themes, id, index, is_real_data,url, is_private, etc
Node_values-> Values of the node_properties instances. Ex: John, Maria, João (those are names) 
relation_types-> (Name of relations) -> has_theme, use_dataset, hold, has_team_memember, etc.
'''

#Node_label_entities
node_label_entity = agent.new_entity('node_label_entities', entries=node_label_entries)

#Node_properties_entities
node_property_entities = agent.new_entity('node_property_entities', entries=node_property_entries)

#relation_type
relations_entities = agent.new_entity('relations_entities', entries=relations_entries)
        
#Node_values_entities (only textual and boolean ones)
node_values_entities = agent.new_entity('node_values_entities', entries=node_values_entries)


In [23]:
len(node_values_entries)

38456

### Define agent intents
There is only a single intent (the user always want to ask the llm)

In [24]:
# INTENTS
llm_intent = agent.new_intent('llm_intent', [
    "Find NODE_LABEL that RELATION to NODE_LABEL2 where NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Which NODE_LABEL with NODE_PROPERTY OPERATOR NODE_VALUE RELATION NODE_LABEL2?",
    "Retrieve NODE_LABEL that RELATION NODE_LABEL2 and satisfy NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Get all NODE_LABEL connected to NODE_LABEL2 via RELATION with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Show me NODE_LABEL that RELATION NODE_LABEL2 where NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Find NODE_LABEL linked to NODE_LABEL2 through RELATION if NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Give me all NODE_LABEL that RELATION NODE_LABEL2 and meet NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "List NODE_LABEL that RELATION NODE_LABEL2 with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Which NODE_LABEL RELATION NODE_LABEL2 when NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE?",
    "Find NODE_LABEL related to NODE_LABEL2 via RELATION with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE LOGIC_OPERATOR NODE_LABEL2's NODE_PROPERTY OPERATOR NODE_VALUE2.",
    "Show NODE_LABEL that RELATION NODE_LABEL2 and fulfill conditions on NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Retrieve NODE_LABEL that RELATION NODE_LABEL2 with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE LOGIC_OPERATOR NODE_LABEL2's NODE_PROPERTY OPERATOR NODE_VALUE2.",
    "Get NODE_LABEL that RELATION NODE_LABEL2 where NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE LOGIC_OPERATOR NODE_LABEL2's NODE_PROPERTY OPERATOR NODE_VALUE2.",
    "Which NODE_LABEL that RELATION NODE_LABEL2 have NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE?",
    "Find NODE_LABEL with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE that RELATION NODE_LABEL2.",
    "What are the NODE_LABEL that RELATION NODE_LABEL2 where NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE LOGIC_OPERATOR NODE_LABEL2's NODE_PROPERTY OPERATOR NODE_VALUE2?",
    "List NODE_LABEL that RELATION NODE_LABEL2 with both NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE LOGIC_OPERATOR NODE_LABEL2's NODE_PROPERTY OPERATOR NODE_VALUE2.",
    "Show all NODE_LABEL that RELATION NODE_LABEL2 and match NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE.",
    "Can you find NODE_LABEL that RELATION NODE_LABEL2, considering NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE?",
    "Retrieve NODE_LABEL with NODE_LABEL's NODE_PROPERTY OPERATOR NODE_VALUE that RELATION NODE_LABEL2.",
    "Find all NODE_LABEL2 that are connected to NODE_LABEL.",
    "Which NODE_LABEL are linked to NODE_LABEL2?",
    "Show all NODE_LABEL that have a RELATION with NODE_LABEL2.",
    "Retrieve NODE_LABEL that have a direct connection to NODE_LABEL2.",
    "List all NODE_LABEL that are connected to NODE_LABEL2 via any RELATION.",
    "Find NODE_LABEL that share any relationship with NODE_LABEL2.",
    "Which NODE_LABEL have any kind of connection with NODE_LABEL2?",
    "Show me the connections between NODE_LABEL and NODE_LABEL2.",
    "Get all NODE_LABEL that are connected to NODE_LABEL2 regardless of RELATION.",
    "Find all pairs of NODE_LABEL and NODE_LABEL2 that are linked.",
    "What are the relationships between NODE_LABEL and NODE_LABEL2?",
    "Give me all relationships between NODE_LABEL and NODE_LABEL2.",
    "Which NODE_LABEL are associated with NODE_LABEL2?",
    "Show NODE_LABEL that have an unspecified connection to NODE_LABEL2.",
    "Find any NODE_LABEL that link to NODE_LABEL2.",
    "Who did this.",
    "Retrieve all connections between NODE_LABEL and NODE_LABEL2."])
llm_intent.parameter('node_label', 'NODE_LABEL', node_label_entity)
llm_intent.parameter('node_label2', 'NODE_LABEL2', node_label_entity)
llm_intent.parameter('node_property', 'NODE_PROPERTY', node_property_entities)
llm_intent.parameter('node_property2', 'NODE_PROPERTY2', node_property_entities)
llm_intent.parameter('node_value', 'NODE_VALUE', node_values_entities)
llm_intent.parameter('node_value2', 'NODE_VALUE2', node_values_entities)
llm_intent.parameter('relation', 'RELATION', relations_entities)
llm_intent.parameter('any', 'ANY', any_entity) #refers to AD_HOC sensitive entities
llm_intent.parameter('any2', 'ANY2', any_entity) #refers to AD_HOC sensitive entities


### Define state bodies

In [25]:
def s0_body(session: Session):
    #print('What is your next question?')
    print('What is your next question?')

s0.set_body(s0_body)
s0.when_intent_matched_go_to(llm_intent, llm_state)

In [26]:
from typing import List
import re
def extract_bracketed_entities(sentence: str) -> List[str]:
    """
    Extract entities wrapped in square brackets [...] from the sentence.
    Returns a list of unique values (without brackets).
    """
    return list(re.findall(r'\[([^\[\]]+)\]', sentence))

In [27]:
extract_bracketed_entities('[Joe Thomas] appears in which [movies] [bla]')

['Joe Thomas', 'movies', 'bla']

In [28]:
from besser.agent.nlp.llm.llm_openai_api import LLMOpenAI
# Create the LLM
gpt = LLMOpenAI(
    agent=agent,
    name='gpt-5',#'gpt-4o-mini',
    parameters={},
    num_previous_messages=1
)

def llm_body(session: Session): #approximately 1370 input tokens per question and < 40 output tokens per query . This corresponds to less than 2.3 cents per 100 queries (gpt-4o-mini).
    
    original_sentence = session._message
    log_question.append(original_sentence)
    ner_sentence = session.predicted_intent.matched_sentence
    log_ner.append(ner_sentence)
    
    predicted_intent: IntentClassifierPrediction = session.predicted_intent
    
    possible_entities = {}                                                                                         

     #Check for AD_HOC sensitive entities
    ad_hoc = predicted_intent.get_parameter('ad_hoc')
    ad_hoc2 = predicted_intent.get_parameter('ad_hoc2')
    if ad_hoc is not None and ad_hoc.value is not None: 
        # Do not use ad_hoc.value because it may have been stemmed
        ad_hoc_values = extract_bracketed_entities(original_sentence)
        possible_entities['AD_HOC'] = ad_hoc_values 
        possible_entities['ad_hoc'] = ad_hoc_values

    #Check from sensitive entities in the KG
    node_value = predicted_intent.get_parameter('node_value')
    node_value2 = predicted_intent.get_parameter('node_value2')
    if node_value is not None and node_value.value is not None: 
        possible_entities['NODE_VALUES_ENTITIES'] = [node_value.value]
        possible_entities['node_values_entities'] = [node_value.value]
        if node_value2 is not None and node_value2.value is not None: 
            possible_entities['NODE_VALUES_ENTITIES'].append(node_value2.value)
            possible_entities['node_values_entities'].append(node_value2.value)

    update_entities = {}
    
    if experiment_type != 'baf_no_synonyms':
        #check for node labels synonyms
        node_label = predicted_intent.get_parameter('node_label')
        node_label2 = predicted_intent.get_parameter('node_label2')
        node_label3 = predicted_intent.get_parameter('node_label3')
        if node_label is not None and node_label.value is not None: 
            update_entities['NODE_LABEL_ENTITIES'] = [node_label.value]
            update_entities['node_label_entities'] = [node_label.value]
            if node_label2 is not None and node_label2.value is not None: 
                update_entities['NODE_LABEL_ENTITIES'].append(node_label2.value)
                update_entities['node_label_entities'].append(node_label2.value)
                if node_label3 is not None and node_label3.value is not None: 
                    update_entities['NODE_LABEL_ENTITIES'].append(node_label3.value)
                    update_entities['node_label_entities'].append(node_label3.value)
    
        #check for node property synonyms
        node_property = predicted_intent.get_parameter('node_property')
        node_property2 = predicted_intent.get_parameter('node_property2')
        node_property3 = predicted_intent.get_parameter('node_property3')
        if node_property is not None and node_property.value is not None: 
            update_entities['NODE_PROPERTY_ENTITIES'] = [node_property.value]
            update_entities['node_property_entities'] = [node_property.value]
            if node_property2 is not None and node_property2.value is not None: 
                update_entities['NODE_PROPERTY_ENTITIES'].append(node_property2.value)
                update_entities['node_property_entities'].append(node_property2.value)
                if node_property3 is not None and node_property3.value is not None: 
                    update_entities['NODE_PROPERTY_ENTITIES'].append(node_property3.value)
                    update_entities['node_property_entities'].append(node_property3.value)
    
        #check for relations synonyms
        relation = predicted_intent.get_parameter('relation')
        relation2 = predicted_intent.get_parameter('relation2')
        if relation is not None and relation.value is not None: 
            update_entities['RELATION_ENTITIES'] = [relation.value]
            update_entities['relation_entities'] = [relation.value]
            if relation2 is not None and relation2.value is not None: 
                update_entities['RELATION_ENTITIES'].append(relation2.value)
                update_entities['relation_entities'].append(relation2.value)

    #Replace values from user message by placeholder before passing them to LLM.
    sentence = replace_placeholders(original_sentence, ner_sentence, possible_entities, update_entities) 
    last_message = sentence
    log_mask.append(last_message)

    if experiment_type == 'gpt':
        last_message = original_sentence
        
    gpt_query = f"""
    You are a tool that transforms natural language questions into Cypher queries.  
    The graph structure is: {graph_structure}.  
    
    Your task: Generate a single Cypher query that best represents the question: {last_message}.  
    
    STRICT RULES (must always be followed):
    1. Output **only** the Cypher query (no explanations, no extra text, no formatting).  
    2. Never include relationship types or directions.  
       - Always write relationships as `-[r]-`.  
       - Forbidden: `-[r]->`, `<-[r]-`, `[:RELATION_TYPE]`.  
       - Wrong: `(a)-[r]->(b)` → Correct: `(a)-[r]-(b)`  
    3. Always restrict nodes by labels **as they appear in `graph_structure`**.  
       - If a node label exists in `graph_structure`, you must include it (e.g. `(m:Movie)`, `(w:Writer)`).  
       - Do not invent or guess labels.  
       - Do not omit labels when they are defined in `graph_structure`.  
    4. Never return whole nodes. Always return a property and remove duplicates:  
       - Example: `RETURN DISTINCT w.name`  
    5. Example of bad vs good:  
    - Bad (labels missing): MATCH (m)-[r]-(w) WHERE m.name = 'AD_HOC' RETURN DISTINCT w.name
    - Good (labels enforced from graph structure): MATCH (m:Movie)-[r]-(w:Writer) WHERE toLower(m.name) = toLower('AD_HOC') RETURN DISTINCT w.name  

    Failure to follow these rules will result in an invalid answer.    
    """
   
    
    answer = gpt.predict(gpt_query)
    
    log_gpt.append(answer)
    answer = replace_placeholders_back(answer, possible_entities)
    log_baf.append(answer)
    return
    
llm_state.set_body(llm_body)
llm_state.go_to(s0)

# Train and Init agent

In [29]:
agent.run(sleep=False, train=True)

2026-01-14 10:56:03,387 - INFO - Stemmer added: english
2026-01-14 10:56:03,430 - INFO - metaqa__agent training started
2026-01-14 10:56:08,291 - INFO - NER successfully trained.
2026-01-14 10:56:09,961 - INFO - Intent classifier in s0 successfully trained.
2026-01-14 10:56:09,962 - INFO - metaqa__agent training finished
2026-01-14 10:56:09,963 - INFO - metaqa__agent's WebSocketPlatform starting at ws://localhost:8765


In [30]:
from besser.agent.platforms.websocket.websocket_platform import WebSocketPlatform
agent.get_or_create_session(session_id='0', platform=WebSocketPlatform)

2026-01-14 10:56:09,969 - INFO - [s0] Running body s0_body


What is your next question?


# Prepare experiment

In [31]:
#Supress logging 
import logging
lg = logging.getLogger("BESSER Agentic Framework")
lg.disabled = True

In [32]:
def clean_answer(ans):
     # If exists, remove the leading ```cypher\n 
    without_prefix = ans.replace("```cypher\n", "", 1)
    # Keep only up to the first newline
    query = without_prefix.split("\n", 1)[0]
    return query
    
def run_query(query):
    # Capture the print output
    agent.receive_message(session_id='0', message=query)
    result = log_baf[-1].replace("\n", " ")
    return clean_answer(result)


In [33]:
from tqdm import tqdm

def run_baf(questions, experiment_type):
    baf_answers = []

    for question in tqdm(questions):
        
        result = run_query(question)
        baf_answers.append(result)

        with open(experiment_type+".tsv", "a") as f:
            f.write(log_question[-1].replace("\n", " ") + "\t" + log_ner[-1].replace("\n", " ") + "\t" + log_mask[-1].replace("\n", " ") + "\t" + log_gpt[-1].replace("\n", " ") + "\t" + log_baf[-1].replace("\n", " ")+ "\n")

    return baf_answers

In [34]:
def run_cypher(baf_answers, URI, AUTH):
    #Run CYPHER queries
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    results = []
    for answer in tqdm(baf_answers):
        try:
            records, summary, keys = driver.execute_query(answer,database_=DATABASE_NAME)
            results.append((records, summary, keys))
        except:
            results.append(([], None, None))
    driver.close()

    parsed_results = []
    for result in results:
        data = result[0]  # this is a list of dictionaries
        parsed_results.append([v for d in data for v in d.values()])
    
    return parsed_results

### Test

# Run

In [35]:
#Those log_lists are used to store intermediate results (used to analyze final outut) 
log_question = [] 
log_ner = []
log_mask = []
log_gpt = []
log_baf = []
gpt_queries = []

############################### experiemnt_type options ############################
# baf_complete: use full pipeline
# baf_no_synonyms: do not substitute synonym for node labels, node properties, and relations
# gpt: do not mask questions 

sample_questions = sample_questions
sample_answers = sample_answers

experiment_type = 'baf_complete' 
baf_answers = run_baf(sample_questions, experiment_type)

  0%|▏                                                                                | 1/503 [00:23<3:13:56, 23.18s/it]

What is your next question?


  0%|▎                                                                                | 2/503 [00:41<2:49:31, 20.30s/it]

What is your next question?


  1%|▍                                                                                | 3/503 [00:58<2:36:42, 18.80s/it]

...

What is your next question?


100%|██████████████████████████████████████████████████████████████████████████████▊| 502/503 [3:09:57<00:40, 40.89s/it]

What is your next question?


100%|███████████████████████████████████████████████████████████████████████████████| 503/503 [3:10:29<00:00, 22.72s/it]

What is your next question?


In [36]:
import pickle

with open(experiment_type+'.pickle', 'wb') as handle:
    pickle.dump(baf_answers, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [37]:
parsed_results = run_cypher(log_baf, URI, AUTH)

 16%|████████████▋                                                                     | 78/503 [00:02<00:13, 31.38it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (m) { ... }', position=<SummaryInputPosition line=1, column=258, offset=257>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 257, 'line': 1, 'column': 258}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (m:Movie) WHERE toLower(m.name) = toLower('The Butcher Boy') CALL { WITH m OPTIONAL MATCH (m)-[r]-(d:Director) RETURN collect(DISTINCT d.name) AS directors } CALL { WITH m OPTIONAL MATCH (m)-[r]-(w:Writer) RETURN

# Calculate results

In [38]:
from typing import List

def accuracy_score(ground_truth: List[List[str]], predictions: List[List[str]]):
    wrong = []
    correct = 0
    total = len(ground_truth)
    
    for i, (gt, pred) in enumerate(zip(ground_truth, predictions)):
        # Flatten pred if it contains lists
        flattened = [x for item in pred for x in (item if isinstance(item, list) else [item])]
        
        if set(gt) == set(flattened):
            correct += 1
        else:
            wrong.append([i, gt, pred])

    acc = correct / total if total > 0 else 0.0
    return acc, wrong

In [ ]:
acc, wrong_answers = accuracy_score(sample_answers, parsed_results)
print("Accuracy:", acc)

In [40]:
#Compare answers individually
index=19

print("log_question: ", log_question[index])
print("log_ner: ", log_ner[index])
print("log_mask: ", log_mask[index])
print("log_gpt: ", log_gpt[index])
print("log_baf: ", log_baf[index])
print("parsed_results: ", parsed_results[index])
print("ground truth: ", sample_answers[index])

log_question:  what is a film directed by [Eddie Murphy]
log_ner:  what is a NODE_LABEL_ENTITIES NODE_PROPERTY_ENTITIES AD_HOC
log_mask:  what is a Movie directed_by AD_HOC
log_gpt:  MATCH (m:Movie)-[r]-(d:Director) WHERE toLower(d.name) = toLower('AD_HOC') RETURN DISTINCT m.name
log_baf:  MATCH (m:Movie)-[r]-(d:Director) WHERE toLower(d.name) = toLower('Eddie Murphy') RETURN DISTINCT m.name
parsed_results:  ['Harlem Nights']
ground truth:  ['Harlem Nights']


In [41]:
wrong_answers

[[30,
  ['Bing Crosby', 'Louis Armstrong', 'Madge Evans', 'Edith Fellows'],
  ['Madge Evans',
   'Bing Crosby',
   'Steve Martin',
   'Louis Armstrong',
   'Bernadette Peters',
   'Edith Fellows',
   'Jessica Harper']],
 [57, ['Documentary'], []],
 [64, ['Drama', 'Crime'], ['Drama', 'Comedy', 'Crime', 'Action']],
 [68, ['English', 'French'], ['English']],
 [77, ['basketball'], ['basketball', 'Documentary']],
 [78, ['bd-r', 'spring break'], ['Comedy', 'Romance']],
 [80,
  ['jason segel', 'judy greer', 'mark duplass'],
  ['jason segel', 'judy greer', 'mark duplass', 'Drama', 'Comedy']],
 [81,
  ['neil jordan'],
  ['The Butcher Boy',
   ['1997'],
   None,
   None,
   ['Drama'],
   [],
   ['Neil Jordan'],
   ['Neil Jordan'],
   ['Eamonn Owens']]],
 [83, ['franchise'], ['Horror']],
 [85, ['john ford', 'silent'], []],
 [86, ['don siegel'], ['Drama']],
 [87, ['paul dano'], []],
 [89,
  ['Ariane Mnouchkine'],
  ['Ariane Mnouchkine', 'Grégoire Vigneron', 'Laurent Tirard']],
 [94,
  ['Tony Richa

In [42]:
question = '[Joe Thomas] appears in which movies'
ner_question = 'AD_HOC appear in which NODE_LABEL_ENTITIES'
possible_entities = {'AD_HOC': ['Grégoir Colin'], 'ad_hoc': ['Grégoir Colin']}
update_entities = {'NODE_LABEL_ENTITIES': ['Movie'], 'node_label_entities': ['Movie']}

In [43]:
sentence = replace_placeholders(question, ner_question, possible_entities, update_entities)
sentence

'AD_HOC appears in which Movie'